# Marketing ROI & Budget Reallocation — ETL Pipeline
### Part B: Data Architecture & ETL
**Project:** E-Commerce Marketing Analytics Capstone  
**Objective:** Load, clean, and transform raw data into three curated fact tables for attribution + regression analysis.  
**Attribution Method:** Last-Touch (session channel/campaign → order)  
**Output Tables:**
- `fact_sessions.csv` — 1 row per session
- `fact_campaign_daily.csv` — 1 row per date per campaign
- `fact_channel_daily.csv` — 1 row per date per channel

---

## 1. Import Libraries

In [17]:
import pandas as pd
import numpy as np
import json
import os
import warnings
warnings.filterwarnings("ignore")

print("Libraries loaded")
print(f"  pandas  : {pd.__version__}")
print(f"  numpy   : {np.__version__}")

Libraries loaded
  pandas  : 2.3.3
  numpy   : 2.3.5


## 2. Load Raw Files
All 7 source files are loaded here.  
`sessions1.csv` and `sessions2.csv` are **combined into a single DataFrame** as they represent one split dataset.

In [18]:
print("Loading raw files...")

users       = pd.read_csv("input/users.csv")
campaigns   = pd.read_csv("input/campaigns.csv")
spend       = pd.read_csv("input/ad_spend_daily.csv")
orders      = pd.read_csv("input/orders.csv")
order_items = pd.read_csv("input/order_items.csv")
sessions 			= pd.read_csv("input/sessions.csv")

with open("input/products.json") as f:
    products = pd.DataFrame(json.load(f))

print(f"{'File':<20} {'Rows':>8}")
print("-" * 30)
for name, df in [("users", users), ("campaigns", campaigns), ("ad_spend_daily", spend),
                  ("sessions", sessions), ("orders", orders),
                  ("order_items", order_items), ("products", products)]:
    print(f"{name:<20} {len(df):>8,}")

Loading raw files...
File                     Rows
------------------------------
users                   2,600
campaigns                  40
ad_spend_daily          8,461
sessions              659,793
orders                 16,225
order_items            40,456
products                  240


## 3. Quick Data Preview

In [19]:
print("── sessions sample ──")
display(sessions.head(3))
print("\n── orders sample ──")
display(orders.head(3))
print("\n── ad_spend sample ──")
display(spend.head(3))

── sessions sample ──


,session_id,user_id,session_ts,device,channel,campaign_id
0,S00000001,U001881,2025-07-04T16:50:13,mobile,search,C001
1,S00000002,U001667,2025-07-04T06:09:01,web,search,C001
2,S00000003,U001894,2025-07-04T03:57:40,web,search,C001



── orders sample ──


,order_id,session_id,user_id,order_ts,gross_amount,discount_amount,shipping_amount,net_amount
0,O00000001,S00000002,U001667,2025-07-04T06:17:29,12869.16,2261.63,0.0,10607.53
1,O00000002,S00000018,U001077,2025-07-04T21:41:35,12537.13,1328.78,0.0,11208.35
2,O00000003,S00000088,U002260,2025-07-04T05:26:18,7783.09,892.60,0.0,6890.49



── ad_spend sample ──


,date,day_idx,promo_flag,campaign_id,channel,spend,impressions,clicks
0,2025-07-04,0,1,C001,search,26312.21,100619,2173
1,2025-07-04,0,1,C002,search,21647.84,87070,3103
2,2025-07-04,0,1,C003,search,22354.06,92590,1931


## 4. Clean & Standardize — Sessions
**Steps applied:**
- Normalize column names (strip + lowercase)
- Normalize `channel` and `device` values to lowercase
- Parse `session_ts` as datetime
- Deduplicate on `session_id`

In [20]:
# Normalize columns
sessions.columns    = sessions.columns.str.strip().str.lower()
sessions['channel']     = sessions['channel'].str.strip().str.lower()
sessions['device']      = sessions['device'].str.strip().str.lower()
sessions['campaign_id'] = sessions['campaign_id'].astype(str).str.strip()
sessions['session_ts']  = pd.to_datetime(sessions['session_ts'], errors='coerce')

# Deduplicate
n_before = len(sessions)
sessions = sessions.drop_duplicates(subset='session_id')
n_removed = n_before - len(sessions)

print(f"Sessions before dedup : {n_before:,}")
print(f"Duplicates removed    : {n_removed:,}")
print(f"Sessions after dedup  : {len(sessions):,}")
print(f"\nColumn dtypes:\n{sessions.dtypes}")

Sessions before dedup : 659,793
Duplicates removed    : 1,973
Sessions after dedup  : 657,820

Column dtypes:
session_id             object
user_id                object
session_ts     datetime64[ns]
device                 object
channel                object
campaign_id            object
dtype: object


## 5. Clean & Standardize — Orders
**Steps applied:**
- Normalize column names
- Parse `order_ts` as datetime
- Deduplicate on `order_id`
- **Flag & cap revenue outliers** using 3×IQR upper fence (outliers are flagged and capped for KPI calculations but original values are preserved)

In [21]:
orders.columns     = orders.columns.str.strip().str.lower()
orders['order_ts'] = pd.to_datetime(orders['order_ts'], errors='coerce')

# Deduplicate
n_before = len(orders)
orders   = orders.drop_duplicates(subset='order_id')
print(f"Orders deduped: {n_before - len(orders):,} removed → {len(orders):,} remain")

# Outlier detection — 3×IQR upper fence
q1, q3 = orders['gross_amount'].quantile([0.25, 0.75])
iqr     = q3 - q1
fence   = q3 + 3 * iqr

orders['revenue_outlier_flag']  = (orders['gross_amount'] > fence).astype(int)
orders['gross_amount_capped']   = orders['gross_amount'].clip(upper=fence)
orders['net_amount_capped']     = orders['net_amount'].clip(upper=fence)

n_outliers = orders['revenue_outlier_flag'].sum()
print(f"Revenue outlier fence : ₹{fence:,.2f}")
print(f"Outliers flagged      : {n_outliers}")
print(f"  (kept in data; capped values used for KPI calculations)")

display(orders.describe()[['gross_amount','gross_amount_capped','net_amount','net_amount_capped']])

Orders deduped: 32 removed → 16,193 remain
Revenue outlier fence : ₹27,393.99
Outliers flagged      : 1
  (kept in data; capped values used for KPI calculations)


,gross_amount,gross_amount_capped,net_amount,net_amount_capped
count,16193.000000,16193.000000,16193.000000,16193.000000
mean,6744.800721,6744.781043,6487.639285,6367.947532
min,256.710000,256.710000,270.540000,270.540000
25%,3327.630000,3327.630000,3119.670000,3119.670000
50%,6048.620000,6048.620000,5650.070000,5650.070000
75%,9344.220000,9344.220000,8761.770000,8761.770000
max,27712.630000,27393.990000,170924.880000,27393.990000
std,4338.932792,4338.838415,5646.204081,4177.588056


## 6. Clean & Standardize — Other Tables
Applying standard cleaning (column normalization, deduplication, type parsing) to `order_items`, `products`, `users`, `campaigns`, and `ad_spend_daily`.

In [22]:
# Order items
order_items.columns = order_items.columns.str.strip().str.lower()
order_items         = order_items.drop_duplicates(subset=['order_id', 'product_id'])

# Products
products.columns    = products.columns.str.strip().str.lower()

# Users
users.columns       = users.columns.str.strip().str.lower()
users['signup_date']= pd.to_datetime(users['signup_date'], errors='coerce')
users['segment']    = users['segment'].str.strip().str.lower()
users               = users.drop_duplicates(subset='user_id')

# Campaigns
campaigns.columns   = campaigns.columns.str.strip().str.lower()
campaigns['channel']= campaigns['channel'].str.strip().str.lower()

# Ad Spend
spend.columns       = spend.columns.str.strip().str.lower()
spend['date']       = pd.to_datetime(spend['date'], errors='coerce')
spend['channel']    = spend['channel'].str.strip().str.lower()
n_before            = len(spend)
spend               = spend.drop_duplicates(subset=['date', 'campaign_id'])

# Handle missing spend/clicks/impressions
for col in ['spend', 'clicks', 'impressions']:
    spend[col] = pd.to_numeric(spend[col], errors='coerce').fillna(0)

print(f"Spend deduped : {n_before - len(spend):,} removed → {len(spend):,} remain")
print("Missing values filled with 0 for: spend, clicks, impressions")
print("\n All tables cleaned successfully")

Spend deduped : 21 removed → 8,440 remain
Missing values filled with 0 for: spend, clicks, impressions

 All tables cleaned successfully


## 7. Compute COGS Proxy per Order
Using `products.json` (which contains `cost` per product) joined with `order_items`, we compute the **total cost of goods sold (COGS)** per order, then derive a **margin proxy**:

`margin_proxy = net_amount − total_cogs`

In [23]:
# Join order_items with product cost data
oi_prod = order_items.merge(
    products[['product_id', 'cost', 'category']],
    on='product_id', how='left'
)
oi_prod['line_cogs'] = oi_prod['quantity'] * oi_prod['cost']

# Aggregate COGS to order level
order_cogs = (
    oi_prod.groupby('order_id')['line_cogs']
    .sum()
    .reset_index(name='total_cogs')
)

# Merge back to orders
orders = orders.merge(order_cogs, on='order_id', how='left')
orders['total_cogs']   = orders['total_cogs'].fillna(0)
orders['margin_proxy'] = orders['net_amount'] - orders['total_cogs']

print(f"Orders with COGS data : {(orders['total_cogs'] > 0).sum():,}")
print(f"Avg Margin Proxy      : ₹{orders['margin_proxy'].mean():,.2f}")
print(f"Avg Margin Rate       : {(orders['margin_proxy'] / orders['net_amount']).mean():.1%}")

display(orders[['order_id','gross_amount','net_amount','total_cogs','margin_proxy']].head(5))

Orders with COGS data : 16,193
Avg Margin Proxy      : ₹2,450.44
Avg Margin Rate       : 36.1%


,order_id,gross_amount,net_amount,total_cogs,margin_proxy
0,O00000001,12869.16,10607.53,8745.94,1861.59
1,O00000002,12537.13,11208.35,6565.58,4642.77
2,O00000003,7783.09,6890.49,4278.12,2612.37
3,O00000004,12458.61,11953.35,7466.36,4486.99
4,O00000005,10186.43,9785.49,6170.38,3615.11


## 8. Output 1 — `fact_sessions.csv`
**1 row per session.** Includes:
- Session metadata (id, user, timestamp, device, channel, campaign)
- `is_new_user` flag — user signed up within 30 days before the session
- Purchase flag + `order_id` (if the session converted)
- Revenue fields: `gross`, `discount`, `net` (0 if no purchase)
- `total_cogs` and `margin_proxy`
- `session_to_order_mins` — time from session start to order placement

In [24]:
# Merge user signup date onto sessions
sessions = sessions.merge(users[['user_id', 'signup_date']], on='user_id', how='left')

# is_new_user: signed up within 30 days before session
sessions['days_since_signup'] = (sessions['session_ts'] - sessions['signup_date']).dt.days
sessions['is_new_user']       = (sessions['days_since_signup'] <= 30).astype(int)

# Prepare order data to join onto sessions (last-touch: 1 session = 1 order max)
order_join = orders[[
    'order_id', 'session_id', 'order_ts',
    'gross_amount', 'discount_amount', 'net_amount',
    'gross_amount_capped', 'net_amount_capped',
    'total_cogs', 'margin_proxy', 'revenue_outlier_flag'
]].copy()
order_join['purchased'] = 1

# Left join: sessions get order info if they converted
sessions = sessions.merge(order_join, on='session_id', how='left')

# Fill nulls for non-converting sessions
fill_zero = ['purchased','gross_amount','discount_amount','net_amount',
             'gross_amount_capped','net_amount_capped',
             'total_cogs','margin_proxy','revenue_outlier_flag']
for col in fill_zero:
    sessions[col] = sessions[col].fillna(0)

sessions['purchased']            = sessions['purchased'].astype(int)
sessions['revenue_outlier_flag'] = sessions['revenue_outlier_flag'].astype(int)
sessions['order_id']             = sessions['order_id'].fillna('')

# Session-to-order time (minutes)
sessions['session_to_order_mins'] = (
    (sessions['order_ts'] - sessions['session_ts']).dt.total_seconds() / 60
).round(2)

# Final column selection
fact_sessions = sessions[[
    'session_id', 'user_id', 'session_ts', 'device', 'channel', 'campaign_id',
    'is_new_user', 'purchased', 'order_id',
    'gross_amount', 'discount_amount', 'net_amount',
    'gross_amount_capped', 'net_amount_capped',
    'total_cogs', 'margin_proxy', 'revenue_outlier_flag',
    'session_to_order_mins'
]].copy()

fact_sessions.to_csv("data/fact_sessions.csv", index=False)
print(f" fact_sessions.csv saved")
print(f"   Rows       : {len(fact_sessions):,}")
print(f"   Purchases  : {fact_sessions['purchased'].sum():,}")
print(f"   Overall CVR: {fact_sessions['purchased'].mean():.2%}")
display(fact_sessions.head(3))

 fact_sessions.csv saved
   Rows       : 657,820
   Purchases  : 16,193
   Overall CVR: 2.46%


,session_id,user_id,session_ts,device,channel,campaign_id,is_new_user,purchased,order_id,gross_amount,discount_amount,net_amount,gross_amount_capped,net_amount_capped,total_cogs,margin_proxy,revenue_outlier_flag,session_to_order_mins
0,S00000001,U001881,2025-07-04 16:50:13,mobile,search,C001,1,0,,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0,NaN
1,S00000002,U001667,2025-07-04 06:09:01,web,search,C001,1,1,O00000001,12869.16,2261.63,10607.53,12869.16,10607.53,8745.94,1861.59,0,8.47
2,S00000003,U001894,2025-07-04 03:57:40,web,search,C001,1,0,,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0,NaN


## 9. Output 2 — `fact_campaign_daily.csv`
**1 row per date per campaign.** Includes:
- Spend, impressions, clicks (from `ad_spend_daily`)
- Attributed sessions, orders, revenue, margin (from last-touch attribution via `fact_sessions`)
- Derived KPIs: `CPC`, `CTR`, `CVR`, `ROAS`, `CAC Proxy`
- `promo_flag` — carried from ad spend data

In [25]:
# Add session date
fact_sessions['session_date'] = pd.to_datetime(fact_sessions['session_ts']).dt.normalize()

# Aggregate sessions + orders by campaign × date (last-touch attribution)
camp_sess = (
    fact_sessions
    .groupby(['session_date', 'campaign_id'])
    .agg(
        attributed_sessions=('session_id',      'count'),
        attributed_orders  =('purchased',        'sum'),
        attributed_revenue =('net_amount_capped','sum'),
        attributed_margin  =('margin_proxy',     'sum')
    )
    .reset_index()
    .rename(columns={'session_date': 'date'})
)

# Get authoritative channel from campaigns master (overrides spend channel if different)
camp_ch = campaigns[['campaign_id', 'channel']].drop_duplicates()
spend2  = spend.merge(camp_ch, on='campaign_id', how='left', suffixes=('_s', '_c'))
spend2['channel'] = spend2['channel_c'].fillna(spend2['channel_s'])
spend2  = spend2.drop(columns=['channel_s', 'channel_c'], errors='ignore')

# Merge spend with attribution
fcd = spend2.merge(camp_sess, on=['date', 'campaign_id'], how='left')
for col in ['attributed_sessions','attributed_orders','attributed_revenue','attributed_margin']:
    fcd[col] = fcd[col].fillna(0)

# Derived KPIs
fcd['cpc']       = np.where(fcd['clicks']>0,               fcd['spend']/fcd['clicks'],                               np.nan)
fcd['ctr']       = np.where(fcd['impressions']>0,           fcd['clicks']/fcd['impressions'],                         np.nan)
fcd['cvr']       = np.where(fcd['attributed_sessions']>0,   fcd['attributed_orders']/fcd['attributed_sessions'],      np.nan)
fcd['roas']      = np.where(fcd['spend']>0,                 fcd['attributed_revenue']/fcd['spend'],                   np.nan)
fcd['cac_proxy'] = np.where(fcd['attributed_orders']>0,     fcd['spend']/fcd['attributed_orders'],                    np.nan)

for col in ['cpc','ctr','cvr','roas','cac_proxy']:
    fcd[col] = fcd[col].round(4)

fact_campaign_daily = fcd[[
    'date','campaign_id','channel','spend','impressions','clicks',
    'attributed_sessions','attributed_orders','attributed_revenue','attributed_margin',
    'cpc','ctr','cvr','roas','cac_proxy','promo_flag'
]]

fact_campaign_daily.to_csv("data/fact_campaign_daily.csv", index=False)
print(f"fact_campaign_daily.csv saved")
print(f"   Rows     : {len(fact_campaign_daily):,}")
print(f"   Campaigns: {fact_campaign_daily['campaign_id'].nunique()}")
print(f"   Date range: {fact_campaign_daily['date'].min().date()} → {fact_campaign_daily['date'].max().date()}")
display(fact_campaign_daily.head(3))

fact_campaign_daily.csv saved
   Rows     : 8,440
   Campaigns: 40
   Date range: 2025-07-04 → 2026-01-30


,date,campaign_id,channel,spend,impressions,clicks,attributed_sessions,attributed_orders,attributed_revenue,attributed_margin,cpc,ctr,cvr,roas,cac_proxy,promo_flag
0,2025-07-04,C001,search,26312.21,100619,2173,140.0,3.0,28706.37,9116.73,12.1087,0.0216,0.0214,1.0910,8770.7367,1
1,2025-07-04,C002,search,21647.84,87070,3103,140.0,4.0,17676.27,6007.30,6.9764,0.0356,0.0286,0.8165,5411.9600,1
2,2025-07-04,C003,search,22354.06,92590,1931,140.0,9.0,55752.12,20995.88,11.5764,0.0209,0.0643,2.4940,2483.7844,1


## 10. Output 3 — `fact_channel_daily.csv`
**1 row per date per channel.** Aggregated from `fact_campaign_daily`. Includes:
- Total spend, attributed orders/revenue/margin
- Control variables: `day_of_week`, `day_name`, `week_index`, `promo_flag`
- Derived KPIs: `ROAS`, `CAC Proxy`

> `promo_flag` definition: carried from `ad_spend_daily.csv` (provided in source data).  
> `week_index`: integer week number from observation start date (0 = first week).

In [26]:
# Aggregate from campaign-daily to channel-daily
ch = (
    fact_campaign_daily
    .groupby(['date','channel'])
    .agg(
        total_spend       =('spend',               'sum'),
        attributed_orders =('attributed_orders',    'sum'),
        attributed_revenue=('attributed_revenue',   'sum'),
        attributed_margin =('attributed_margin',    'sum'),
        promo_flag        =('promo_flag',           'max')
    )
    .reset_index()
)

ch['date']        = pd.to_datetime(ch['date'])
ch['day_of_week'] = ch['date'].dt.dayofweek          # 0 = Monday
ch['day_name']    = ch['date'].dt.day_name()
ch['week_index']  = ((ch['date'] - ch['date'].min()).dt.days // 7)

ch['roas']      = np.where(ch['total_spend']>0,       ch['attributed_revenue']/ch['total_spend'],      np.nan)
ch['cac_proxy'] = np.where(ch['attributed_orders']>0, ch['total_spend']/ch['attributed_orders'],        np.nan)
for col in ['roas','cac_proxy']:
    ch[col] = ch[col].round(4)

fact_channel_daily = ch[[
    'date','channel','total_spend','attributed_orders','attributed_revenue',
    'attributed_margin','day_of_week','day_name','week_index','promo_flag','roas','cac_proxy'
]]

fact_channel_daily.to_csv("data/fact_channel_daily.csv", index=False)
print(f"fact_channel_daily.csv saved")
print(f"   Rows    : {len(fact_channel_daily):,}")
print(f"   Channels: {fact_channel_daily['channel'].nunique()}")
display(fact_channel_daily.head(3))

fact_channel_daily.csv saved
   Rows    : 1,055
   Channels: 5


,date,channel,total_spend,attributed_orders,attributed_revenue,attributed_margin,day_of_week,day_name,week_index,promo_flag,roas,cac_proxy
0,2025-07-04,email,19566.32,20.0,123248.30,41412.15,4,Friday,0,1,6.2990,978.3160
1,2025-07-04,organic,5380.51,14.0,64098.51,17576.61,4,Friday,0,1,11.9131,384.3221
2,2025-07-04,paid_social,157932.61,30.0,152446.28,44192.84,4,Friday,0,1,0.9653,5264.4203


## 11. ETL Summary & Validation

In [27]:
total_spend   = fact_campaign_daily['spend'].sum()
total_revenue = fact_sessions['net_amount_capped'].sum()
total_orders  = fact_sessions['purchased'].sum()
total_margin  = fact_sessions['margin_proxy'].sum()
blended_roas  = total_revenue / total_spend

print("=" * 50)
print("  ETL PIPELINE — FINAL SUMMARY")
print("=" * 50)
print(f"  Total Spend         : ₹{total_spend:>14,.0f}")
print(f"  Total Revenue       : ₹{total_revenue:>14,.0f}")
print(f"  Total Margin Proxy  : ₹{total_margin:>14,.0f}")
print(f"  Total Orders        : {total_orders:>15,}")
print(f"  Blended ROAS        : {blended_roas:.2f}x")
print(f"  Blended CVR         : {total_orders / len(fact_sessions):.2%}")
print(f"  Channels            : {fact_channel_daily['channel'].nunique()}")
print(f"  Campaigns           : {fact_campaign_daily['campaign_id'].nunique()}")
print(f"  Date Range          : {fact_channel_daily['date'].min().date()} → {fact_channel_daily['date'].max().date()}")
print("=" * 50)

print("Output files saved to: data")
print("fact_sessions.csv")
print("fact_campaign_daily.csv")
print("fact_channel_daily.csv")

  ETL PIPELINE — FINAL SUMMARY
  Total Spend         : ₹    74,360,800
  Total Revenue       : ₹   103,116,174
  Total Margin Proxy  : ₹    39,679,981
  Total Orders        :          16,193
  Blended ROAS        : 1.39x
  Blended CVR         : 2.46%
  Channels            : 5
  Campaigns           : 40
  Date Range          : 2025-07-04 → 2026-01-30
Output files saved to: data
fact_sessions.csv
fact_campaign_daily.csv
fact_channel_daily.csv


## 12. Quick Channel Performance Preview

In [28]:
channel_summary = (
    fact_channel_daily
    .groupby('channel')
    .agg(
        total_spend       =('total_spend',       'sum'),
        attributed_revenue=('attributed_revenue','sum'),
        attributed_orders =('attributed_orders', 'sum'),
    )
    .assign(
        roas      = lambda d: (d['attributed_revenue'] / d['total_spend']).round(2),
        cac_proxy = lambda d: (d['total_spend'] / d['attributed_orders']).round(0),
        spend_share   = lambda d: (d['total_spend'] / d['total_spend'].sum() * 100).round(1),
        revenue_share = lambda d: (d['attributed_revenue'] / d['attributed_revenue'].sum() * 100).round(1)
        
    )
    .sort_values('roas', ascending=False)
    .reset_index()
)

channel_summary['attributed_orders'] = channel_summary['attributed_orders'].astype('int64')

channel_summary.columns = [
    'Channel','Total Spend (₹)','Revenue (₹)','Orders',
    'ROAS','CAC Proxy (₹)','Spend Share %','Revenue Share %'
]
display(channel_summary.style
        .format({
            'Total Spend (₹)': '₹{:,.0f}',
            'Revenue (₹)':     '₹{:,.0f}',
            'ROAS':             '{:.2f}x',
            'CAC Proxy (₹)':   '₹{:,.0f}',
            'Spend Share %':    '{:.1f}%',
            'Revenue Share %':  '{:.1f}%'
        })
        .background_gradient(subset=['ROAS'], cmap='RdYlGn')
        .set_caption("Channel Performance Summary (Last-Touch Attribution)")
)

,Channel,Total Spend (₹),Revenue (₹),Orders,ROAS,CAC Proxy (₹),Spend Share %,Revenue Share %
0,organic,"₹958,408","₹12,277,062",1999,12.81x,₹479,1.3%,11.9%
1,email,"₹3,813,282","₹19,224,697",3003,5.04x,"₹1,270",5.1%,18.6%
2,search,"₹34,333,236","₹44,730,137",6681,1.30x,"₹5,139",46.2%,43.4%
3,referral,"₹6,669,378","₹8,507,492",1307,1.28x,"₹5,103",9.0%,8.3%
4,paid_social,"₹28,586,496","₹18,376,787",3203,0.64x,"₹8,925",38.4%,17.8%
